# Coincidence Analysis
Ch0 = Alpha | Ch1 = Gamma | TDC Ch0 = Timing

In [ ]:
import numpy as np
import glob
import plotly.graph_objects as go

# Load all files with unique event IDs
files = sorted(glob.glob('run_*.npy'))
arrays = []
offset = 0
for f in files:
    d = np.load(f).copy()
    d['event'] = d['event'] + offset
    offset = int(d['event'].max()) + 1
    arrays.append(d)
    print(f'{f}: {len(d):,} hits')
data = np.concatenate(arrays)
print(f'\nTotal: {len(data):,} hits')

In [ ]:
# Build per-event arrays: for each event, grab alpha/gamma/tdc if present
# This is the ONLY way to do coincidences — match by event ID
from collections import defaultdict

evt = defaultdict(dict)
for row in data:
    eid = int(row['event'])
    s, c, v = int(row['slot']), int(row['channel']), int(row['value'])
    evt[eid][(s, c)] = v

# Extract matched arrays
all_alpha, all_gamma, all_tdc = [], [], []
for eid, hits in evt.items():
    all_alpha.append(hits.get((19, 0), -1))
    all_gamma.append(hits.get((19, 1), -1))
    all_tdc.append(hits.get((21, 0), -1))

all_alpha = np.array(all_alpha)
all_gamma = np.array(all_gamma)
all_tdc   = np.array(all_tdc)

has_a = all_alpha >= 0
has_g = all_gamma >= 0
has_t = all_tdc >= 0

print(f'Total events:       {len(evt):,}')
print(f'Has alpha:          {has_a.sum():,}')
print(f'Has gamma:          {has_g.sum():,}')
print(f'Has TDC:            {has_t.sum():,}')
print(f'Has alpha+gamma:    {(has_a & has_g).sum():,}')
print(f'Has alpha+gamma+tdc:{(has_a & has_g & has_t).sum():,}')

## 1. All spectra (no gate)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=all_alpha[has_a], nbinsx=512, name='Alpha (Ch0)', marker_color='orange'))
fig.update_layout(title=f'Alpha — {has_a.sum():,} hits', template='plotly_dark', height=350)
fig.show()

fig = go.Figure()
fig.add_trace(go.Histogram(x=all_gamma[has_g], nbinsx=512, name='Gamma (Ch1)', marker_color='cyan'))
fig.update_layout(title=f'Gamma — {has_g.sum():,} hits', template='plotly_dark', height=350)
fig.show()

fig = go.Figure()
fig.add_trace(go.Histogram(x=all_tdc[has_t], nbinsx=512, name='TDC (Ch0)', marker_color='lime'))
fig.update_layout(title=f'TDC — {has_t.sum():,} hits', template='plotly_dark', height=350)
fig.show()

## 2. Gate on Alpha → see Gamma and TDC

In [ ]:
# ===== ADJUST =====
A_LO, A_HI = 1900, 1960
# ==================

agate = has_a & (all_alpha >= A_LO) & (all_alpha <= A_HI)
print(f'Alpha gate [{A_LO}-{A_HI}]: {agate.sum():,} events')

# Gamma with alpha gate
sel = agate & has_g
fig = go.Figure()
fig.add_trace(go.Histogram(x=all_gamma[has_g], nbinsx=512, name='All', marker_color='rgba(0,255,255,0.3)'))
fig.add_trace(go.Histogram(x=all_gamma[sel],   nbinsx=512, name='Gated', marker_color='cyan'))
fig.update_layout(title=f'Gamma with Alpha gate [{A_LO}-{A_HI}] — {sel.sum():,} events',
                  template='plotly_dark', height=400, barmode='overlay')
fig.show()

# TDC with alpha gate
sel_t = agate & has_t
fig = go.Figure()
fig.add_trace(go.Histogram(x=all_tdc[has_t],   nbinsx=512, name='All', marker_color='rgba(0,255,0,0.3)'))
fig.add_trace(go.Histogram(x=all_tdc[sel_t],   nbinsx=512, name='Gated', marker_color='lime'))
fig.update_layout(title=f'TDC with Alpha gate [{A_LO}-{A_HI}] — {sel_t.sum():,} events',
                  template='plotly_dark', height=400, barmode='overlay')
fig.show()

## 3. Gate on Gamma → see Alpha and TDC

In [ ]:
# ===== ADJUST =====
G_LO, G_HI = 100, 400
# ==================

ggate = has_g & (all_gamma >= G_LO) & (all_gamma <= G_HI)
print(f'Gamma gate [{G_LO}-{G_HI}]: {ggate.sum():,} events')

# Alpha with gamma gate
sel = ggate & has_a
fig = go.Figure()
fig.add_trace(go.Histogram(x=all_alpha[has_a], nbinsx=512, name='All', marker_color='rgba(255,165,0,0.3)'))
fig.add_trace(go.Histogram(x=all_alpha[sel],   nbinsx=512, name='Gated', marker_color='orange'))
fig.update_layout(title=f'Alpha with Gamma gate [{G_LO}-{G_HI}] — {sel.sum():,} events',
                  template='plotly_dark', height=400, barmode='overlay')
fig.show()

# TDC with gamma gate
sel_t = ggate & has_t
fig = go.Figure()
fig.add_trace(go.Histogram(x=all_tdc[has_t],   nbinsx=512, name='All', marker_color='rgba(0,255,0,0.3)'))
fig.add_trace(go.Histogram(x=all_tdc[sel_t],   nbinsx=512, name='Gated', marker_color='lime'))
fig.update_layout(title=f'TDC with Gamma gate [{G_LO}-{G_HI}] — {sel_t.sum():,} events',
                  template='plotly_dark', height=400, barmode='overlay')
fig.show()

## 4. Double gate (Alpha AND Gamma) → see TDC

In [ ]:
# ===== ADJUST =====
A_LO2, A_HI2 = 1900, 1960
G_LO2, G_HI2 = 100, 400
# ==================

dgate = (has_a & (all_alpha >= A_LO2) & (all_alpha <= A_HI2) &
         has_g & (all_gamma >= G_LO2) & (all_gamma <= G_HI2))
dgate_t = dgate & has_t

print(f'Double gate: Alpha [{A_LO2}-{A_HI2}] AND Gamma [{G_LO2}-{G_HI2}]')
print(f'  Events: {dgate.sum():,}')
print(f'  With TDC: {dgate_t.sum():,}')

fig = go.Figure()
fig.add_trace(go.Histogram(x=all_tdc[has_t],    nbinsx=512, name='All', marker_color='rgba(0,255,0,0.3)'))
fig.add_trace(go.Histogram(x=all_tdc[dgate_t],  nbinsx=512, name='Double gated', marker_color='lime'))
fig.update_layout(title=f'TDC with double gate — {dgate_t.sum():,} events',
                  template='plotly_dark', height=400, barmode='overlay')
fig.show()

## 5. 2D: Alpha vs Gamma

In [ ]:
both = has_a & has_g
fig = go.Figure(go.Histogram2d(
    x=all_alpha[both], y=all_gamma[both],
    nbinsx=256, nbinsy=256,
    colorscale='Hot', reversescale=True,
))
fig.update_layout(
    title=f'Alpha vs Gamma — {both.sum():,} events',
    xaxis_title='Alpha (Ch0)', yaxis_title='Gamma (Ch1)',
    height=600, width=700, template='plotly_dark',
)
fig.show()